# Leitura de Múltiplos Parquets com PySpark

Notebook para ler todos os arquivos Parquet de uma pasta, detectar se contêm dados JSON e expandir automaticamente.

## 1. Configurar SparkSession

Importar PySpark e criar uma SparkSession.

In [17]:
from pyspark.sql import SparkSession

# Criar SparkSession
spark = SparkSession.builder \
    .appName("Leitura Parquet") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark version: {spark.version}")
spark

Spark version: 3.4.0


## 2. Resumo dos Arquivos

Tabela resumo mostrando quais arquivos têm JSON e quantos registros.

In [18]:
import glob
import os

# Pasta com os arquivos Parquet
pasta_parquet = "quickbooks_data/daily_incremental/agedpayabledetail"

# Listar todos os arquivos .parquet
arquivos = glob.glob(os.path.join(pasta_parquet, "*.parquet"))
print(f"Arquivos encontrados: {len(arquivos)}\n")

# Dicionário para guardar os DataFrames
dataframes = {}

for arquivo in arquivos:
    nome = os.path.basename(arquivo)
    print(f"--- {nome} ---")
    df = spark.read.parquet(arquivo)
    registros = df.count()
    print(f"  Registros: {registros}")
    print(f"  Colunas: {df.columns}")

    # Detectar se alguma coluna contém JSON (string que começa com '{' ou '[')
    tem_json = False
    colunas_json = []
    for c in df.columns:
        # Verificar tipo string
        tipo = dict(df.dtypes).get(c)
        if tipo == "string" and registros > 0:
            amostra = df.select(c).filter(f"`{c}` IS NOT NULL").first()
            if amostra:
                valor = amostra[0]
                if isinstance(valor, str) and len(valor) > 0 and valor.strip()[0] in ('{', '['):
                    tem_json = True
                    colunas_json.append(c)

    if tem_json:
        print(f"  ✅ Contém JSON nas colunas: {colunas_json}")
    else:
        print(f"  ❌ Não contém dados JSON")

    dataframes[nome] = {"df": df, "tem_json": tem_json, "colunas_json": colunas_json}
    print()

Arquivos encontrados: 1

--- AgedPayableDetail_20260207170626.parquet ---
  Registros: 1
  Colunas: ['Header', 'Columns', 'Rows']
  ✅ Contém JSON nas colunas: ['Header', 'Columns', 'Rows']



import pandas as pd

resumo = []
for nome, info in dataframes.items():
    resumo.append({
        "Arquivo": nome,
        "Registros": info["df"].count(),
        "Colunas": len(info["df"].columns),
        "Tem JSON": "Sim" if info["tem_json"] else "Não",
        "Colunas JSON": ", ".join(info["colunas_json"]) if info["colunas_json"] else "-"
    })

df_resumo = pd.DataFrame(resumo)
df_resumo

## 3. Verificar se as Colunas JSON são Iguais

Comparar os schemas JSON de todos os arquivos que contêm JSON para verificar compatibilidade.

In [19]:
from pyspark.sql.functions import from_json, schema_of_json, col

# Coletar schemas JSON de cada arquivo
schemas_json = {}
for nome, info in dataframes.items():
    if not info["tem_json"]:
        continue
    df = info["df"]
    for col_json in info["colunas_json"]:
        amostra = df.select(col_json).filter(f"`{col_json}` IS NOT NULL").first()[0]
        json_schema = schema_of_json(amostra)
        # Parsear para obter o StructType real
        df_temp = df.withColumn("parsed", from_json(col(col_json), json_schema)).select("parsed.*")
        schemas_json[nome] = {
            "colunas": sorted(df_temp.columns),
            "schema": df_temp.schema,
            "dtypes": sorted(df_temp.dtypes, key=lambda x: x[0])
        }

# Comparar todos os schemas entre si
arquivos_com_json = list(schemas_json.keys())
print(f"Arquivos com JSON: {len(arquivos_com_json)}\n")

if len(arquivos_com_json) < 2:
    print("⚠️ Apenas 1 arquivo com JSON encontrado, nada para comparar.")
else:
    referencia = arquivos_com_json[0]
    colunas_ref = set(schemas_json[referencia]["colunas"])
    todos_iguais = True

    for arquivo in arquivos_com_json[1:]:
        colunas_atual = set(schemas_json[arquivo]["colunas"])

        print(f"Comparando: {referencia} vs {arquivo}")

        # Colunas iguais?
        if colunas_ref == colunas_atual:
            print(f"  ✅ Mesmas colunas ({len(colunas_ref)} colunas)")
        else:
            todos_iguais = False
            apenas_ref = colunas_ref - colunas_atual
            apenas_atual = colunas_atual - colunas_ref
            em_comum = colunas_ref & colunas_atual
            print(f"  ❌ Colunas diferentes!")
            print(f"     Em comum: {len(em_comum)} → {sorted(em_comum)}")
            if apenas_ref:
                print(f"     Só em {referencia}: {sorted(apenas_ref)}")
            if apenas_atual:
                print(f"     Só em {arquivo}: {sorted(apenas_atual)}")

        # Tipos iguais?
        dtypes_ref = schemas_json[referencia]["dtypes"]
        dtypes_atual = schemas_json[arquivo]["dtypes"]
        if dtypes_ref == dtypes_atual:
            print(f"  ✅ Tipos de dados idênticos")
        else:
            todos_iguais = False
            print(f"  ❌ Tipos de dados diferem:")
            for (col_r, tipo_r), (col_a, tipo_a) in zip(dtypes_ref, dtypes_atual):
                if col_r == col_a and tipo_r != tipo_a:
                    print(f"     Coluna '{col_r}': {tipo_r} → {tipo_a}")
        print()

    if todos_iguais:
        print("✅ RESULTADO: Todos os arquivos JSON têm exatamente as mesmas colunas e tipos!")
    else:
        print("❌ RESULTADO: Existem diferenças entre os schemas JSON dos arquivos.")

Arquivos com JSON: 1

⚠️ Apenas 1 arquivo com JSON encontrado, nada para comparar.


In [20]:
import json
import pandas as pd
from IPython.display import display

def extrair_linhas(row_data, grupo_pai=None):
    """Extrai recursivamente as linhas de dados de um relatório QuickBooks."""
    linhas = []
    rows = row_data.get("Row", [])

    for row in rows:
        # Identificar grupo (Header com sub-Rows)
        nome_grupo = grupo_pai
        if "Header" in row and "Rows" in row:
            # É um grupo - pegar o nome do grupo do primeiro ColData
            col_data = row["Header"].get("ColData", [])
            if col_data and col_data[0].get("value"):
                nome_grupo = col_data[0]["value"]
            # Processar sub-linhas recursivamente
            linhas.extend(extrair_linhas(row["Rows"], nome_grupo))
        elif "ColData" in row:
            # É uma linha de dados
            valores = [item.get("value", "") for item in row["ColData"]]
            linhas.append({"grupo": nome_grupo, "valores": valores})
        elif "Header" in row and "ColData" in row.get("Header", {}):
            # Header sem sub-rows (pode ser linha simples)
            col_data = row["Header"].get("ColData", [])
            valores = [item.get("value", "") for item in col_data]
            linhas.append({"grupo": nome_grupo, "valores": valores})

    return linhas

# Transformar cada arquivo com formato de relatório em tabela
for nome, info in dataframes.items():
    if not info["tem_json"]:
        print(f"--- {nome}: Sem JSON ---")
        info["df"].show(5, truncate=False)
        continue

    row = info["df"].first()

    # Extrair nomes das colunas
    cols_json = json.loads(row["Columns"])
    nomes_colunas = [c["ColTitle"] for c in cols_json.get("Column", [])]
    print(f"=== {nome} ===")
    print(f"Colunas do relatório: {nomes_colunas}")

    # Extrair header/metadados
    header = json.loads(row["Header"])
    print(f"Relatório: {header.get('ReportName', '?')} | Período: {header.get('EndPeriod', '?')}")

    # Extrair linhas de dados
    rows_json = json.loads(row["Rows"])
    linhas = extrair_linhas(rows_json)
    print(f"Linhas de dados extraídas: {len(linhas)}")

    if linhas:
        # Montar DataFrame com coluna de grupo + colunas do relatório
        dados = []
        for linha in linhas:
            registro = {"Grupo": linha["grupo"]}
            for i, col_nome in enumerate(nomes_colunas):
                registro[col_nome] = linha["valores"][i] if i < len(linha["valores"]) else ""
            dados.append(registro)

        df_tabela = pd.DataFrame(dados)
        info["df_tabela"] = df_tabela
        print(f"Total de colunas na tabela: {len(df_tabela.columns)}\n")
        display(df_tabela)
    else:
        print("Nenhuma linha de dados encontrada.\n")

=== AgedPayableDetail_20260207170626.parquet ===
Colunas do relatório: ['Date', 'Transaction Type', 'Num', 'Vendor', 'Due Date', 'Past Due', 'Amount', 'Open Balance']
Relatório: AgedPayableDetail | Período: 2026-02-07
Linhas de dados extraídas: 5
Total de colunas na tabela: 9



,Grupo,Date,Transaction Type,Num,Vendor,Due Date,Past Due,Amount,Open Balance
0,91 or more days past due,2025-10-07,Bill,,PG&E,2025-11-06,93,86.44,86.44
1,61 - 90 days past due,2025-11-21,Bill,,Robertson & Associates,2025-11-21,78,315.00,315.00
2,61 - 90 days past due,2025-11-21,Bill,,Norton Lumber and Building Materials,2025-11-21,78,205.00,205.00
3,61 - 90 days past due,2025-11-14,Bill,,Brosnahan Insurance Agency,2025-11-24,75,241.23,241.23
4,31 - 60 days past due,2025-11-19,Bill,,Diego's Road Warrior Bodyshop,2025-12-19,50,755.00,755.00


In [21]:
import json
from IPython.display import display

# Explorar a estrutura JSON de cada coluna
for nome, info in dataframes.items():
    if not info["tem_json"]:
        continue
    print(f"=== {nome} ===\n")
    row = info["df"].first()
    for c in info["colunas_json"]:
        val = row[c]
        if val:
            parsed = json.loads(val)
            print(f"--- Coluna: {c} ---")
            if isinstance(parsed, list):
                print(f"  Tipo: Array com {len(parsed)} elementos")
                print(f"  Primeiro elemento: {json.dumps(parsed[0], indent=2, ensure_ascii=False)[:500]}")
            elif isinstance(parsed, dict):
                print(f"  Tipo: Objeto com chaves: {list(parsed.keys())}")
                print(f"  Valor: {json.dumps(parsed, indent=2, ensure_ascii=False)[:500]}")
            print()

=== AgedPayableDetail_20260207170626.parquet ===

--- Coluna: Header ---
  Tipo: Objeto com chaves: ['Time', 'ReportName', 'EndPeriod', 'Currency', 'Option']
  Valor: {
  "Time": "2026-02-07T12:06:49-08:00",
  "ReportName": "AgedPayableDetail",
  "EndPeriod": "2026-02-07",
  "Currency": "USD",
  "Option": [
    {
      "Name": "report_date",
      "Value": "2026-02-07"
    },
    {
      "Name": "NoReportData",
      "Value": "false"
    }
  ]
}

--- Coluna: Columns ---
  Tipo: Objeto com chaves: ['Column']
  Valor: {
  "Column": [
    {
      "ColTitle": "Date",
      "ColType": "Date",
      "MetaData": [
        {
          "Name": "ColKey",
          "Value": "tx_date"
        }
      ]
    },
    {
      "ColTitle": "Transaction Type",
      "ColType": "String",
      "MetaData": [
        {
          "Name": "ColKey",
          "Value": "txn_type"
        }
      ]
    },
    {
      "ColTitle": "Num",
      "ColType": "String",
      "MetaData": [
        {
          "Name": "Co

In [22]:
# Encerrar SparkSession
spark.stop()
print("SparkSession encerrada.")

SparkSession encerrada.
